# Permanent Phase Validation Notebook

This notebook is the single reusable test harness for future optimization phases.
Current run: Phase 1B train-only character n-gram range tuning.
No candidate is copied into production unless its Public LB score improves.


In [ ]:
from pathlib import Path
import sys
ROOT = Path("/kaggle/input")
print("Permanent validation notebook ready", flush=True)


In [ ]:
"""Phase 1: pool-aware, prompt-grouped validation for Name That LLM.

This is a diagnostic phase. It does not write the production Kaggle notebook.
It compares train-only and train+pool SVM fits under the same prompt-held-out
validation folds, so external-data gains are measured on the distribution that
will actually be used at inference time.

Example:
  python src/phase1_pool_aware_validation.py \
    --train data/raw/train.csv \
    --pool data/raw/pool_v1.parquet \
    --out reports/phase1_pool_validation.json
"""
from __future__ import annotations

import argparse
import glob
import json
import re
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold
from sklearn.svm import LinearSVC

VIEWS = [
    ("char", 1, 5, 300_000, 1.0),
    ("word", 1, 2, 220_000, 0.8),
    ("shape", 2, 6, 80_000, 0.5),
]


def shape_text(text: str) -> str:
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", text)


def load_pool(path: Path, labels: set[str]) -> pd.DataFrame:
    pool = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)
    required = {"text", "model"}
    missing = required - set(pool.columns)
    if missing:
        raise ValueError(f"Pool is missing columns: {sorted(missing)}")
    pool = pool.dropna(subset=["text", "model"]).copy()
    pool["text"] = pool["text"].astype(str)
    pool["model"] = pool["model"].astype(str)
    pool = pool[pool["model"].isin(labels)].copy()
    words = pool["text"].str.split().str.len()
    pool = pool[words.between(40, 250)].copy()
    return pool.drop_duplicates("text").reset_index(drop=True)


def make_splitter(train: pd.DataFrame, y: np.ndarray, n_splits: int):
    groups = train["prompt_id"].astype(str) if "prompt_id" in train else np.arange(len(train))
    try:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
        splits = list(splitter.split(train, y, groups))
    except Exception:
        splitter = GroupKFold(n_splits=min(n_splits, len(np.unique(groups))))
        splits = list(splitter.split(train, y, groups))
    return groups, splits


def transform(texts: Iterable[str]):
    return [str(x) for x in texts]


def fit_view_matrices(train_text: list[str], valid_text: list[str]):
    left, right = [], []
    for view, lo, hi, max_features, weight in VIEWS:
        analyzer = "char" if view in {"char", "shape"} else "word"
        prep = shape_text if view == "shape" else (lambda x: x)
        opts = dict(
            analyzer=analyzer,
            ngram_range=(lo, hi),
            lowercase=False,
            sublinear_tf=True,
            min_df=2,
            max_features=max_features,
            dtype=np.float32,
        )
        if analyzer == "word":
            opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
        vectorizer = TfidfVectorizer(**opts)
        left.append(vectorizer.fit_transform([prep(x) for x in train_text]) * weight)
        right.append(vectorizer.transform([prep(x) for x in valid_text]) * weight)
    return hstack(left, format="csr"), hstack(right, format="csr")


def score_bucketed(y_true, y_pred, texts, labels):
    lengths = pd.Series(texts).str.split().str.len().to_numpy()
    result = {}
    for name, mask in {
        "short_40_60": lengths <= 60,
        "mid_61_120": (lengths >= 61) & (lengths <= 120),
        "mid_121_180": (lengths >= 121) & (lengths <= 180),
        "long_181_plus": lengths >= 181,
    }.items():
        result[name] = (
            float(balanced_accuracy_score(y_true[mask], y_pred[mask]))
            if mask.any() else None
        )
    recalls = recall_score(y_true, y_pred, labels=np.arange(len(labels)), average=None, zero_division=0)
    return {"overall_balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "length_buckets": result,
            "min_class_recall": float(recalls.min()),
            "macro_recall": float(recalls.mean())}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", type=Path, required=True)
    ap.add_argument("--pool", type=Path)
    ap.add_argument("--out", type=Path, default=Path("reports/phase1_pool_validation.json"))
    ap.add_argument("--n-splits", type=int, default=3)
    ap.add_argument("--pool-weights", type=float, nargs="+", default=[0.10, 0.35, 0.85])
    args = ap.parse_args()

    train = pd.read_csv(args.train).dropna(subset=["text", "model"]).copy()
    train["text"] = train["text"].astype(str)
    train["model"] = train["model"].astype(str)
    labels = sorted(train["model"].unique().tolist())
    label_to_idx = {label: i for i, label in enumerate(labels)}
    y = train["model"].map(label_to_idx).to_numpy()
    pool = load_pool(args.pool, set(labels)) if args.pool else pd.DataFrame(columns=["text", "model"])
    groups, splits = make_splitter(train, y, args.n_splits)
    candidates = {"train_only": 1.0}
    candidates.update({f"train_plus_pool_weight_{w:g}": w for w in args.pool_weights})
    fold_records = {name: [] for name in candidates}

    for fold, (itr, iva) in enumerate(splits):
        valid = train.iloc[iva]
        base = train.iloc[itr]
        for name, pool_weight in candidates.items():
            if name == "train_only" or pool.empty:
                fit = base
                sample_weight = None
            else:
                fit = pd.concat([base, pool], ignore_index=True)
                sample_weight = np.r_[np.ones(len(base)), np.full(len(pool), pool_weight)]
            X_fit, X_valid = fit_view_matrices(
                transform(fit["text"]), transform(valid["text"])
            )
            model = LinearSVC(C=0.5, class_weight="balanced", max_iter=5000, random_state=42 + fold)
            model.fit(X_fit, fit["model"].map(label_to_idx).to_numpy(), sample_weight=sample_weight)
            pred = model.predict(X_valid)
            metrics = score_bucketed(
                y[iva], pred, valid["text"].tolist(), labels
            )
            fold_records[name].append({"fold": fold, **metrics, "n_fit": len(fit), "n_pool": len(fit) - len(base)})
            print(f"fold={fold} candidate={name} BA={metrics['overall_balanced_accuracy']:.5f}", flush=True)

    summary = {}
    for name, records in fold_records.items():
        summary[name] = {
            "mean_balanced_accuracy": float(np.mean([r["overall_balanced_accuracy"] for r in records])),
            "std_balanced_accuracy": float(np.std([r["overall_balanced_accuracy"] for r in records])),
            "mean_min_class_recall": float(np.mean([r["min_class_recall"] for r in records])),
            "folds": records,
        }
    result = {
        "phase": "pool-aware prompt-grouped validation",
        "labels": labels,
        "train_rows": len(train),
        "pool_rows_after_filter": len(pool),
        "n_splits": len(splits),
        "group_column": "prompt_id" if "prompt_id" in train else None,
        "candidates": summary,
        "selection_rule": "compare pool-trained candidates against each other; do not compare their CV directly to train-only CV without Public LB confirmation",
    }
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(f"saved {args.out}")




In [ ]:
# Phase 1A: train-only LinearSVC C tuning
# This is intentionally separate from pool diagnostics. It tests only C against
# the 0.582 production baseline under the same prompt-grouped folds.
def run_svc_c_tuning(train_path, out_path, c_values=(0.10, 0.25, 0.50, 1.00, 2.00), n_splits=3):
    train = pd.read_csv(train_path).dropna(subset=["text", "model"]).copy()
    train["text"] = train["text"].astype(str)
    train["model"] = train["model"].astype(str)
    labels = sorted(train["model"].unique().tolist())
    label_to_idx = {label: i for i, label in enumerate(labels)}
    y = train["model"].map(label_to_idx).to_numpy()
    _, splits = make_splitter(train, y, n_splits)
    records = {str(c): [] for c in c_values}
    for fold, (itr, iva) in enumerate(splits):
        base, valid = train.iloc[itr], train.iloc[iva]
        X_fit, X_valid = fit_view_matrices(
            base["text"].tolist(), valid["text"].tolist()
        )
        for c in c_values:
            model = LinearSVC(
                C=float(c), class_weight="balanced", max_iter=5000,
                random_state=42 + fold
            )
            model.fit(X_fit, base["model"].map(label_to_idx).to_numpy())
            pred = model.predict(X_valid)
            metrics = score_bucketed(y[iva], pred, valid["text"].tolist(), labels)
            records[str(c)].append({"fold": fold, **metrics})
            print(f"phase=1A fold={fold} C={c:g} BA={metrics['overall_balanced_accuracy']:.5f}", flush=True)
    summary = {}
    for c, rows in records.items():
        summary[c] = {
            "C": float(c),
            "mean_balanced_accuracy": float(np.mean([r["overall_balanced_accuracy"] for r in rows])),
            "std_balanced_accuracy": float(np.std([r["overall_balanced_accuracy"] for r in rows])),
            "mean_min_class_recall": float(np.mean([r["min_class_recall"] for r in rows])),
            "folds": rows,
        }
    result = {
        "phase": "1A train-only LinearSVC C tuning",
        "train_rows": len(train),
        "n_splits": len(splits),
        "candidates": summary,
        "baseline_C": 0.50,
        "selection_rule": "select only if grouped mean improves and no fold regresses materially; Public LB remains the final gate",
    }
    Path(out_path).write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(f"saved {out_path}", flush=True)


In [ ]:
# Phase 1B: train-only character n-gram range tuning
# Only the character view changes; word/shape views, C=0.50 and all data remain fixed.
def run_char_ngram_tuning(train_path, out_path, ranges=((1,5),(2,5),(3,5),(3,6),(2,6)), n_splits=3):
    global VIEWS
    train = pd.read_csv(train_path).dropna(subset=["text", "model"]).copy()
    train["text"] = train["text"].astype(str)
    train["model"] = train["model"].astype(str)
    labels = sorted(train["model"].unique().tolist())
    label_to_idx = {label: i for i, label in enumerate(labels)}
    y = train["model"].map(label_to_idx).to_numpy()
    _, splits = make_splitter(train, y, n_splits)
    records = {f"char_{lo}_{hi}": [] for lo, hi in ranges}
    old_views = VIEWS
    try:
        for fold, (itr, iva) in enumerate(splits):
            base, valid = train.iloc[itr], train.iloc[iva]
            for lo, hi in ranges:
                VIEWS = [
                    ("char", lo, hi, 300_000, 1.0),
                    ("word", 1, 2, 220_000, 0.8),
                    ("shape", 2, 6, 80_000, 0.5),
                ]
                X_fit, X_valid = fit_view_matrices(
                    base["text"].tolist(), valid["text"].tolist()
                )
                model = LinearSVC(C=0.50, class_weight="balanced", max_iter=5000, random_state=42 + fold)
                model.fit(X_fit, base["model"].map(label_to_idx).to_numpy())
                pred = model.predict(X_valid)
                metrics = score_bucketed(y[iva], pred, valid["text"].tolist(), labels)
                key = f"char_{lo}_{hi}"
                records[key].append({"fold": fold, **metrics})
                print(f"phase=1B fold={fold} char=({lo},{hi}) BA={metrics['overall_balanced_accuracy']:.5f}", flush=True)
    finally:
        VIEWS = old_views
    summary = {}
    for key, rows in records.items():
        summary[key] = {
            "range": key.replace("char_", "").replace("_", "-"),
            "mean_balanced_accuracy": float(np.mean([r["overall_balanced_accuracy"] for r in rows])),
            "std_balanced_accuracy": float(np.std([r["overall_balanced_accuracy"] for r in rows])),
            "mean_min_class_recall": float(np.mean([r["min_class_recall"] for r in rows])),
            "folds": rows,
        }
    result = {
        "phase": "1B train-only character n-gram tuning",
        "train_rows": len(train),
        "n_splits": len(splits),
        "current_range": "1-5",
        "candidates": summary,
        "selection_rule": "select only if grouped mean improves without material fold/class regression; Public LB remains final gate",
    }
    Path(out_path).write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(f"saved {out_path}", flush=True)


In [ ]:
# Legacy Phase 1B sweep is kept above but not executed in this clean-data run.
print("legacy Phase 1B sweep skipped; running clean labelled-data diagnostic", flush=True)


## Clean labelled-answer augmentation diagnostic
This is a private diagnostic only. It audits the attached labelled `passages_clean.csv`, keeps complete 18-model prompt blocks, removes exact overlap with competition train/public pool, applies the existing 40–250 word quality window, and removes external rows whose prompt group is in the validation fold. It reports direct, family, GPT-sibling, and Gemini-sibling balanced accuracy. No result is copied into production automatically.


In [ ]:
# Exact production-style SVC pool-weight diagnostic.
# This remains private; it does not modify llm-gmc.ipynb.
from sklearn.metrics import balanced_accuracy_score

PROD_FUNCTION_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
def prod_function_text(t):
    return " ".join(w.lower() if w.lower() in PROD_FUNCTION_WORDS or not w.isalnum() else "CONTENT" for w in re.findall(r"\w+|[^\w\s]", str(t)))
PROD_VIEWS = [("char", 3, 5, 250_000, 1.0), ("word", 1, 3, 180_000, 0.7), ("shape", 2, 6, 60_000, 0.5), ("function", 1, 3, 50_000, 0.5)]
def fit_prod_matrices(fit_text, valid_text):
    left, right = [], []
    for view, lo, hi, max_features, weight in PROD_VIEWS:
        prep = shape_text if view == "shape" else prod_function_text if view == "function" else (lambda x: str(x))
        analyzer = "char" if view == "shape" else "word" if view == "function" else view
        opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2, max_features=max_features, dtype=np.float32)
        if analyzer == "word": opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
        vec = TfidfVectorizer(**opts)
        left.append(vec.fit_transform([prep(x) for x in fit_text]) * weight)
        right.append(vec.transform([prep(x) for x in valid_text]) * weight)
    return hstack(left, format="csr"), hstack(right, format="csr")
FAMILY_MAP_PROD = {
    "gpt-5.5":"openai", "gpt-5.6-luna":"openai", "gpt-5.6-sol":"openai", "gpt-5.6-terra":"openai", "gpt-6-astra":"openai",
    "gemini-3.1-pro-preview":"google", "gemini-3.6-flash":"google", "gemini-3.7-flash":"google", "gemini-3.8-flash":"google",
    "claude-fable-5":"anthropic", "claude-opus-4-6":"anthropic", "claude-opus-4-8":"anthropic", "claude-opus-5":"anthropic", "claude-sonnet-5":"anthropic",
    "qwen3.8-max":"qwen", "glm-5.3":"zai", "glm-5.3-flash":"zai", "kimi-k3":"moonshot"}
train_hits = sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True)); pool_hits = sorted(glob.glob("/kaggle/input/**/pool_v1.parquet", recursive=True))
assert train_hits and pool_hits, "competition train or pool source missing"
tr = pd.read_csv(train_hits[0]).dropna(subset=["text", "model"]).copy(); pool_prod = pd.read_parquet(pool_hits[0]).dropna(subset=["text", "model"]).copy()
tr["text"] = tr["text"].astype(str); tr["model"] = tr["model"].astype(str); pool_prod["text"] = pool_prod["text"].astype(str); pool_prod["model"] = pool_prod["model"].astype(str)
labels = sorted(tr.model.unique().tolist()); label_to_idx = {x:i for i,x in enumerate(labels)}; y = tr.model.map(label_to_idx).to_numpy()
pool_prod = pool_prod[pool_prod.model.isin(labels)].copy()
print(f"production-style source rows: train={len(tr):,}, pool={len(pool_prod):,}, models={pool_prod.model.nunique()}", flush=True)
_, splits = make_splitter(tr, y, 3)
weights = [0.25, 0.50, 1.00]
records = {str(w): [] for w in weights}
for fold, (itr, iva) in enumerate(splits, 1):
    base, valid = tr.iloc[itr], tr.iloc[iva]
    for w in weights:
        fit = pd.concat([base[["text", "model"]], pool_prod[["text", "model"]]], ignore_index=True)
        sw = np.r_[np.ones(len(base), dtype=np.float32), np.full(len(pool_prod), w, dtype=np.float32)]
        Xf, Xv = fit_prod_matrices(fit.text.tolist(), valid.text.tolist())
        clf = LinearSVC(C=0.5, class_weight="balanced", max_iter=5000, random_state=700+fold)
        clf.fit(Xf, fit.model.map(label_to_idx).to_numpy(), sample_weight=sw)
        pred_idx = clf.predict(Xv).astype(int); pred = np.asarray(labels, dtype=object)[pred_idx]
        yt = valid.model.to_numpy(); yt_idx = valid.model.map(label_to_idx).to_numpy()
        rec = score_bucketed(yt_idx, pred_idx, valid.text.tolist(), labels)
        for name, labs in {"gpt_sibling":[x for x in labels if x.startswith("gpt-")], "gemini_sibling":[x for x in labels if x.startswith("gemini-")]} .items():
            keep = np.isin(yt, labs); rec[name] = float(balanced_accuracy_score(yt[keep], pred[keep]))
        fam_true = np.array([FAMILY_MAP_PROD[x] for x in yt]); fam_pred = np.array([FAMILY_MAP_PROD[x] for x in pred])
        rec["family_balanced_accuracy"] = float(balanced_accuracy_score(fam_true, fam_pred)); rec.update({"fold":fold,"weight":w})
        records[str(w)].append(rec)
        print(f"prod_pool fold={fold} weight={w:g} BA={rec['overall_balanced_accuracy']:.5f} GPT={rec['gpt_sibling']:.5f} Gemini={rec['gemini_sibling']:.5f} Family={rec['family_balanced_accuracy']:.5f}", flush=True)
summary = {}
for w, rows in records.items():
    summary[w] = {"weight":float(w), "mean_balanced_accuracy":float(np.mean([r["overall_balanced_accuracy"] for r in rows])), "std_balanced_accuracy":float(np.std([r["overall_balanced_accuracy"] for r in rows])), "mean_gpt_sibling":float(np.mean([r["gpt_sibling"] for r in rows])), "mean_gemini_sibling":float(np.mean([r["gemini_sibling"] for r in rows])), "mean_family_balanced_accuracy":float(np.mean([r["family_balanced_accuracy"] for r in rows])), "folds":rows}
best_key=max(summary, key=lambda k: summary[k]["mean_balanced_accuracy"]); base_pub=0.582; svc_fraction=1.021/(1.021+0.226+0.429); uplift=summary[best_key]["mean_balanced_accuracy"]-summary["1.0"]["mean_balanced_accuracy"]
# Conservative transfer: only 25–75% of the SVC-only uplift reaches the full ensemble.
expected = [base_pub + uplift*svc_fraction*r for r in (0.25,0.50,0.75)]
result={"phase":"exact production-style SVC pool-weight diagnostic", "source_rows":{"train":len(tr),"pool":len(pool_prod)}, "candidates":summary, "best_weight":float(best_key), "local_uplift_vs_current_weight_1":float(uplift), "svc_blend_fraction":float(svc_fraction), "expected_public_lb_range":[float(min(expected)),float(max(expected))], "expected_public_lb_center":float(expected[1]), "calibration":"Version 40 Public LB=0.582; conservative 25-75% transfer of SVC-only uplift", "decision":"do not add to LLM GMC unless private uplift is stable and expected range clears 0.582"}
Path("/kaggle/working/production_svc_pool_weight_results.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
print(json.dumps(result, indent=2), flush=True)


In [ ]:
# Disagreement-first audit: compare the current production-style SVC (pool weight 1.0)
# with candidate weights on held-out rows. No production notebook is changed here.
from sklearn.metrics import accuracy_score

def _fit_disagreement_model(fit, Xf, w, fold):
    sw = np.r_[np.ones(len(fit) - len(pool_prod), dtype=np.float32),
               np.full(len(pool_prod), w, dtype=np.float32)]
    m = LinearSVC(C=0.5, class_weight="balanced", max_iter=5000, random_state=900 + fold)
    m.fit(Xf, fit.model.map(label_to_idx).to_numpy(), sample_weight=sw)
    return m

disagreement_records = {"0.25": [], "0.50": []}
for fold, (itr, iva) in enumerate(splits, 1):
    base, valid = tr.iloc[itr], tr.iloc[iva]
    fit = pd.concat([base[["text", "model"]], pool_prod[["text", "model"]]], ignore_index=True)
    Xf, Xv = fit_prod_matrices(fit.text.tolist(), valid.text.tolist())
    current = _fit_disagreement_model(fit, Xf, 1.0, fold)
    current_pred = current.predict(Xv).astype(int)
    y_valid = valid.model.map(label_to_idx).to_numpy()
    current_correct = current_pred == y_valid
    for candidate_w in (0.25, 0.50):
        candidate = _fit_disagreement_model(fit, Xf, candidate_w, fold)
        candidate_pred = candidate.predict(Xv).astype(int)
        changed = candidate_pred != current_pred
        changed_n = int(changed.sum())
        changed_correct = int((candidate_pred[changed] == y_valid[changed]).sum()) if changed_n else 0
        baseline_changed_correct = int((current_pred[changed] == y_valid[changed]).sum()) if changed_n else 0
        rec = {
            "fold": fold, "candidate_pool_weight": candidate_w,
            "validation_rows": int(len(valid)), "changed_rows": changed_n,
            "changed_fraction": float(changed.mean()),
            "current_overall_accuracy": float(current_correct.mean()),
            "candidate_overall_accuracy": float((candidate_pred == y_valid).mean()),
            "current_accuracy_on_changed": float(baseline_changed_correct / changed_n) if changed_n else None,
            "candidate_accuracy_on_changed": float(changed_correct / changed_n) if changed_n else None,
            "changed_row_net_correct": int(changed_correct - baseline_changed_correct),
        }
        disagreement_records[str(candidate_w)].append(rec)
        print(f"disagreement fold={fold} weight={candidate_w:g} changed={changed_n}/{len(valid)} "
              f"current={rec['current_overall_accuracy']:.5f} candidate={rec['candidate_overall_accuracy']:.5f} "
              f"changed_current={rec['current_accuracy_on_changed']} changed_candidate={rec['candidate_accuracy_on_changed']}", flush=True)

disagreement_summary = {}
for w, rows in disagreement_records.items():
    disagreement_summary[w] = {
        "candidate_pool_weight": float(w),
        "total_changed_rows": int(sum(r["changed_rows"] for r in rows)),
        "mean_changed_fraction": float(np.mean([r["changed_fraction"] for r in rows])),
        "mean_current_accuracy": float(np.mean([r["current_overall_accuracy"] for r in rows])),
        "mean_candidate_accuracy": float(np.mean([r["candidate_overall_accuracy"] for r in rows])),
        "mean_current_accuracy_on_changed": float(np.mean([r["current_accuracy_on_changed"] for r in rows if r["current_accuracy_on_changed"] is not None])) if any(r["current_accuracy_on_changed"] is not None for r in rows) else None,
        "mean_candidate_accuracy_on_changed": float(np.mean([r["candidate_accuracy_on_changed"] for r in rows if r["candidate_accuracy_on_changed"] is not None])) if any(r["candidate_accuracy_on_changed"] is not None for r in rows) else None,
        "net_correct_on_changed_rows": int(sum(r["changed_row_net_correct"] for r in rows)),
        "folds": rows,
    }
result_path = Path("/kaggle/working/disagreement_first_results.json")
result_path.write_text(json.dumps({
    "phase": "disagreement-first private audit",
    "reference": "production-style SVC with pool weight 1.0",
    "candidates": disagreement_summary,
    "rule": "candidate must change rows and improve changed-row correctness before LLM GMC test",
}, indent=2), encoding="utf-8")
print(json.dumps(disagreement_summary, indent=2), flush=True)
